![](https://storage.googleapis.com/kaggle-competitions/kaggle/28009/logos/header.png?)

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from sklearn.pipeline import Pipeline


import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout, Embedding,  Flatten
from tensorflow.keras.models import Model, Sequential
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

from tensorflow.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer,  KBinsDiscretizer
from tensorflow import keras
from sklearn import metrics
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split

from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load Dataset

In [2]:
%%time
train = pd.read_csv('../input/tabular-playground-series-nov-2021/train.csv')
test  = pd.read_csv('../input/tabular-playground-series-nov-2021/test.csv')
sub   = pd.read_csv('../input/tabular-playground-series-nov-2021/sample_submission.csv')

CPU times: user 20.2 s, sys: 1.5 s, total: 21.7 s
Wall time: 29.9 s


# Preprocessing

In [3]:
%%time
train['n_missing'] = train.isna().sum(axis=1)
test['n_missing'] = test.isna().sum(axis=1)
train['target'] = train['target'].astype(str)

features = [col for col in train.columns if col not in ['target', 'id']]
pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median',missing_values=np.nan)),
        ("scaler", QuantileTransformer(n_quantiles=64,output_distribution='uniform')),
        ('bin', KBinsDiscretizer(n_bins=64, encode='ordinal',strategy='uniform'))
        ])
train[features] = pipe.fit_transform(train[features])
test[features] = pipe.transform(test[features])

CPU times: user 38.1 s, sys: 1.64 s, total: 39.7 s
Wall time: 39.7 s


# Modeling

In [4]:
model = Sequential([
    Input(train[features].shape[1:]),
    Embedding(input_dim=64, output_dim=4),
    Flatten(),
    Dense(64,  activation='relu'),
    Dropout(0.5),
    Dense(32,  activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid'),
])

auc = tf.keras.metrics.AUC(name='aucroc')
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=5e-4,
        decay_steps = 900,
        decay_rate= 0.9)
callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule)

optimizer = RMSprop(lr=5e-4, rho=0.9, epsilon=1e-08, decay=0.0)

model.compile(loss='binary_crossentropy', optimizer = optimizer, metrics=[auc]) 



User settings:

   KMP_AFFINITY=granularity=fine,verbose,compact,1,0
   KMP_BLOCKTIME=0
   KMP_DUPLICATE_LIB_OK=True
   KMP_INIT_AT_FORK=FALSE
   KMP_SETTINGS=1
   KMP_WARNINGS=0

Effective settings:

   KMP_ABORT_DELAY=0
   KMP_ADAPTIVE_LOCK_PROPS='1,1024'
   KMP_ALIGN_ALLOC=64
   KMP_ALL_THREADPRIVATE=128
   KMP_ATOMIC_MODE=2
   KMP_BLOCKTIME=0
   KMP_CPUINFO_FILE: value is not defined
   KMP_DETERMINISTIC_REDUCTION=false
   KMP_DEVICE_THREAD_LIMIT=2147483647
   KMP_DISP_NUM_BUFFERS=7
   KMP_DUPLICATE_LIB_OK=true
   KMP_ENABLE_TASK_THROTTLING=true
   KMP_FORCE_REDUCTION: value is not defined
   KMP_FOREIGN_THREADS_THREADPRIVATE=true
   KMP_FORKJOIN_BARRIER='2,2'
   KMP_FORKJOIN_BARRIER_PATTERN='hyper,hyper'
   KMP_GTID_MODE=3
   KMP_HANDLE_SIGNALS=false
   KMP_HOT_TEAMS_MAX_LEVEL=1
   KMP_HOT_TEAMS_MODE=0
   KMP_INIT_AT_FORK=true
   KMP_LIBRARY=throughput
   KMP_LOCK_KIND=queuing
   KMP_MALLOC_POOL_INCR=1M
   KMP_NUM_LOCKS_IN_BLOCK=1
   KMP_PLAIN_BARRIER='2,2'
   KMP_PLAIN_BARRIER_P

In [5]:
model.fit(x = np.float32(train[features]), y = np.float32(train.target),
          batch_size = 1024, shuffle = True, epochs = 100,callbacks=[callback])

2021-11-15 14:19:21.547757: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


Epoch 1/100
586/586 [==============================] - 6s 8ms/step - loss: 0.6454 - aucroc: 0.6874
Epoch 2/100
586/586 [==============================] - 5s 9ms/step - loss: 0.6075 - aucroc: 0.7366
Epoch 3/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5961 - aucroc: 0.7437
Epoch 4/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5897 - aucroc: 0.7453
Epoch 5/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5866 - aucroc: 0.7461
Epoch 6/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5853 - aucroc: 0.7467
Epoch 7/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5843 - aucroc: 0.7477
Epoch 8/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5839 - aucroc: 0.7481
Epoch 9/100
586/586 [==============================] - 5s 9ms/step - loss: 0.5832 - aucroc: 0.7492
Epoch 10/100
586/586 [==============================] - 5s 8ms/step - loss: 0.5830 - aucroc: 0.7491
Epoch 11/

In [6]:
sub['target'] = model.predict(np.float32(test[features]))
sub=sub.set_index('id')
sub.to_csv('submission.csv')